# Part 3: Prompt Engineering & Evaluation

In [ ]:
# --- Reconnect to Foundry (run this first after opening a new notebook) ---
import subprocess, sys, json, shutil
from pathlib import Path

def run_cmd(args, check=True):
    exe = shutil.which(args[0])
    if exe: args = [exe] + args[1:]
    result = subprocess.run(args, capture_output=True, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(result.stderr.strip())
    return result

from datetime import date
_user_id = run_cmd(["az", "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"]).stdout.strip()[:6]
SUFFIX = f"{_user_id}-{date.today().strftime('%m%d')}"
RESOURCE_GROUP = f"rg-foundry-workshop-{SUFFIX}"

outputs = json.loads(run_cmd([
    "az", "deployment", "group", "show", "-g", RESOURCE_GROUP, "-n", "main",
    "--query", "properties.outputs", "-o", "json"
]).stdout)

PROJECT_ENDPOINT = outputs["projectEndpoint"]["value"]
MODEL_DEPLOYMENT_NAME = outputs["modelDeploymentName"]["value"]
APP_INSIGHTS_CONN_STR = outputs["appInsightsConnectionString"]["value"]

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, FileSearchTool, WebSearchPreviewTool, FunctionTool
from azure.identity import DefaultAzureCredential, AzureCliCredential

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
openai_client = project_client.get_openai_client()
print(f"✅ Connected to: {PROJECT_ENDPOINT}")
print(f"   Model: {MODEL_DEPLOYMENT_NAME}")

---
## Section 3: Prompt Engineering (~7 min)

The quality of an agent depends heavily on its **instructions** (system prompt).
Key patterns:
- **Persona**: Define who the agent is and its expertise
- **Constraints**: What it should and shouldn't do
- **Output format**: Specify structure (JSON, markdown, etc.)
- **Few-shot examples**: Show desired input/output pairs

In [9]:
# --- 3.1 Basic vs. Engineered Prompt ---
# Compare a vague prompt with a well-engineered one.

QUESTION = "Analyze the potential of AI in quality control for manufacturing."

# Basic prompt
basic_agent = project_client.agents.create_version(
    agent_name="basic-analyst",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions="You are helpful.",
    ),
)
r_basic = openai_client.responses.create(
    input=QUESTION,
    extra_body={"agent_reference": {"name": basic_agent.name, "type": "agent_reference"}},
)

# Engineered prompt
eng_agent = project_client.agents.create_version(
    agent_name="engineered-analyst",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a senior industrial technology analyst with 15 years of experience "
            "in manufacturing and quality assurance.\n\n"
            "RULES:\n"
            "- Structure your analysis with clear headings\n"
            "- Include specific, quantifiable benefits where possible\n"
            "- Address both opportunities AND risks\n"
            "- Keep responses under 300 words\n"
            "- End with 3 actionable recommendations\n\n"
            "FORMAT: Use markdown with ## headings."
        ),
    ),
)
r_eng = openai_client.responses.create(
    input=QUESTION,
    extra_body={"agent_reference": {"name": eng_agent.name, "type": "agent_reference"}},
)

print("=" * 60)
print("BASIC PROMPT RESPONSE:")
print("=" * 60)
print(r_basic.output_text[:500])
print(f"\n... ({len(r_basic.output_text)} chars total)")
print("\n" + "=" * 60)
print("ENGINEERED PROMPT RESPONSE:")
print("=" * 60)
print(r_eng.output_text[:800])
print(f"\n... ({len(r_eng.output_text)} chars total)")


In [10]:
# --- 3.2 Structured JSON Output ---
# Instruct the agent to return valid JSON with a predefined schema.

json_agent = project_client.agents.create_version(
    agent_name="structured-analyst",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a business analyst. Always respond with ONLY valid JSON "
            "(no markdown fences, no extra text) using this schema:\n"
            '{\n'
            '  "executive_summary": "2-3 sentence overview",\n'
            '  "key_findings": ["finding 1", "finding 2", "finding 3"],\n'
            '  "recommendations": ["rec 1", "rec 2"],\n'
            '  "risk_level": "low | medium | high"\n'
            '}'
        ),
    ),
)
r_json = openai_client.responses.create(
    input="Assess the business case for deploying AI-powered visual inspection in a food processing plant.",
    extra_body={"agent_reference": {"name": json_agent.name, "type": "agent_reference"}},
)

print("Raw response:")
print(r_json.output_text)

# Parse and pretty-print
try:
    parsed = json.loads(r_json.output_text)
    print("\n✅ Valid JSON! Parsed fields:")
    for key, value in parsed.items():
        print(f"  {key}: {value}")
except json.JSONDecodeError as e:
    print(f"\n⚠️ JSON parse error: {e}")


In [11]:
# --- 3.3 Few-Shot Prompting ---
# Include example Q&A pairs in the instructions to guide the agent's style.

fewshot_agent = project_client.agents.create_version(
    agent_name="fewshot-advisor",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a concise technical advisor. Follow the style shown in these examples:\n\n"
            "USER: Should we migrate to microservices?\n"
            "ASSISTANT: **Verdict**: Only if your team exceeds 20 engineers.\n"
            "**Why**: Microservices add operational overhead (CI/CD, observability, networking). "
            "Below 20 engineers, a modular monolith delivers the same benefits with less complexity.\n"
            "**Action**: Start with a modular monolith; extract services only when team boundaries demand it.\n\n"
            "USER: Is Kubernetes necessary for our 3-service app?\n"
            "ASSISTANT: **Verdict**: No — use Azure Container Apps instead.\n"
            "**Why**: K8s is overkill for <10 services. Container Apps provides scale-to-zero, "
            "managed ingress, and Dapr integration without cluster management.\n"
            "**Action**: Deploy to Container Apps; re-evaluate K8s at 10+ services.\n\n"
            "Always use this exact format: **Verdict**, **Why**, **Action**."
        ),
    ),
)

r_fs = openai_client.responses.create(
    input="Should we build our own LLM evaluation framework or use an existing one?",
    extra_body={"agent_reference": {"name": fewshot_agent.name, "type": "agent_reference"}},
)
print("🧑 Should we build our own LLM evaluation framework or use an existing one?\n")
print(f"🤖 {r_fs.output_text}")


---
## Section 4: Multi-Step Reasoning & Evaluation (~10 min)

In this section we'll:
1. Build an agent that performs **multi-step reasoning** (analyze → research → synthesize)
2. **Evaluate** agent quality using Microsoft's built-in evaluators

In [12]:
# --- 4.1 Multi-Step Reasoning Agent ---
# This agent analyzes a vague request, researches feasibility, and produces
# a structured requirements document — all in one turn via chain-of-thought prompting.

requirements_agent = project_client.agents.create_version(
    agent_name="requirements-analyst",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a senior business analyst. When given a vague request, follow these steps:\n\n"
            "STEP 1 — ANALYSIS: Identify what's being asked. List assumptions and ambiguities.\n"
            "STEP 2 — CLARIFYING QUESTIONS: List 3-5 questions you would ask the stakeholder.\n"
            "STEP 3 — RESEARCH: Based on your knowledge, assess technical feasibility.\n"
            "STEP 4 — REQUIREMENTS DOCUMENT: Produce a structured output with:\n"
            "  - Objective (1-2 sentences)\n"
            "  - Functional Requirements (numbered list)\n"
            "  - Non-Functional Requirements (numbered list)\n"
            "  - Risks & Mitigations (table format)\n"
            "  - Estimated Complexity: Low | Medium | High\n\n"
            "Show your reasoning at each step. Use markdown formatting."
        ),
        tools=[WebSearchPreviewTool()],
    ),
)

multi_step_query = "We need an AI system that can read our equipment manuals and answer technician questions."

r_ms = openai_client.responses.create(
    input=multi_step_query,
    extra_body={"agent_reference": {"name": requirements_agent.name, "type": "agent_reference"}},
)
print(f"🧑 {multi_step_query}\n")
print(f"🤖 {r_ms.output_text}")

# Save for evaluation
multi_step_response = r_ms.output_text


In [13]:
# --- 4.2 Single-Response Evaluation ---
# Evaluate the agent's output using built-in cloud evaluators via the OpenAI Evals API.

import time
from openai.types.eval_create_params import DataSourceConfigCustom
from openai.types.evals.create_eval_jsonl_run_data_source_param import (
    CreateEvalJSONLRunDataSourceParam,
    SourceFileContent,
    SourceFileContentContent,
    SourceFileID,
)

data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "response": {"type": "string"},
        },
        "required": ["query", "response"],
    },
)

testing_criteria = [
    {
        "type": "azure_ai_evaluator",
        "name": "coherence",
        "evaluator_name": "builtin.coherence",
        "initialization_parameters": {"deployment_name": MODEL_DEPLOYMENT_NAME},
        "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}"},
    },
    {
        "type": "azure_ai_evaluator",
        "name": "relevance",
        "evaluator_name": "builtin.relevance",
        "initialization_parameters": {"deployment_name": MODEL_DEPLOYMENT_NAME},
        "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}"},
    },
    {
        "type": "azure_ai_evaluator",
        "name": "fluency",
        "evaluator_name": "builtin.fluency",
        "initialization_parameters": {"deployment_name": MODEL_DEPLOYMENT_NAME},
        "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}"},
    },
]

eval_obj = openai_client.evals.create(
    name="Multi-Step Response Evaluation",
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,
)

# Run with inline data (the multi-step response from above)
single_run = openai_client.evals.runs.create(
    eval_id=eval_obj.id,
    name="multi-step-check",
    data_source=CreateEvalJSONLRunDataSourceParam(
        type="jsonl",
        source=SourceFileContent(
            type="file_content",
            content=[SourceFileContentContent(item={"query": multi_step_query, "response": multi_step_response})],
        ),
    ),
)

while True:
    single_run = openai_client.evals.runs.retrieve(run_id=single_run.id, eval_id=eval_obj.id)
    if single_run.status in ("completed", "failed"):
        break
    time.sleep(3)

print("Evaluating the multi-step reasoning response...\n")
if single_run.status == "completed":
    items = list(openai_client.evals.runs.output_items.list(run_id=single_run.id, eval_id=eval_obj.id))
    for r in items[0].results:
        print(f"  {r.name}: {r.score:.0f}/5 ({r.label})")
        if r.reason:
            print(f"    Reason: {r.reason[:120]}...")
        print()
else:
    print(f"❌ Evaluation failed: {single_run.status}")

In [ ]:
# --- 4.3 Batch Evaluation with Dataset ---
# Run evaluation across multiple pre-built query/response pairs using the cloud Evals API.
# Results are published to the NEW Foundry portal.

# 1. Upload the evaluation dataset
eval_data_path = str(Path.cwd() / "sample_data" / "eval_dataset.jsonl")
data_id = project_client.datasets.upload_file(
    name="workshop-eval-data",
    version="1",
    file_path=eval_data_path,
).id
print(f"📄 Dataset uploaded: {data_id}")

# 2. Define schema matching our JSONL fields
batch_data_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "response": {"type": "string"},
            "context": {"type": "string"},
        },
        "required": ["query", "response"],
    },
)

# 3. Define evaluators
batch_criteria = [
    {
        "type": "azure_ai_evaluator",
        "name": "coherence",
        "evaluator_name": "builtin.coherence",
        "initialization_parameters": {"deployment_name": MODEL_DEPLOYMENT_NAME},
        "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}"},
    },
    {
        "type": "azure_ai_evaluator",
        "name": "relevance",
        "evaluator_name": "builtin.relevance",
        "initialization_parameters": {"deployment_name": MODEL_DEPLOYMENT_NAME},
        "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}"},
    },
    {
        "type": "azure_ai_evaluator",
        "name": "fluency",
        "evaluator_name": "builtin.fluency",
        "initialization_parameters": {"deployment_name": MODEL_DEPLOYMENT_NAME},
        "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}"},
    },
    {
        "type": "azure_ai_evaluator",
        "name": "groundedness",
        "evaluator_name": "builtin.groundedness",
        "initialization_parameters": {"deployment_name": MODEL_DEPLOYMENT_NAME},
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{item.response}}",
            "context": "{{item.context}}",
        },
    },
]

# 4. Create evaluation + run
batch_eval = openai_client.evals.create(
    name="Workshop Batch Evaluation",
    data_source_config=batch_data_config,
    testing_criteria=batch_criteria,
)

print(f"⏳ Running batch evaluation...\n")
batch_run = openai_client.evals.runs.create(
    eval_id=batch_eval.id,
    name="workshop-batch-run",
    data_source=CreateEvalJSONLRunDataSourceParam(
        type="jsonl",
        source=SourceFileID(type="file_id", id=data_id),
    ),
)

# 5. Poll for completion
while True:
    batch_run = openai_client.evals.runs.retrieve(run_id=batch_run.id, eval_id=batch_eval.id)
    if batch_run.status in ("completed", "failed"):
        break
    print("   Waiting for eval run to complete...")
    time.sleep(5)

if batch_run.status == "completed":
    output_items = list(
        openai_client.evals.runs.output_items.list(run_id=batch_run.id, eval_id=batch_eval.id)
    )
    print(f"=== Results ({len(output_items)} rows) ===")
    for i, item in enumerate(output_items):
        scores = {r.name: f"{r.score:.0f} ({r.label})" for r in item.results}
        print(f"  Row {i+1}: {scores}")

    print(f"\n🔗 View in Foundry Portal: {batch_run.report_url}")
    print("✅ Batch evaluation complete!")
else:
    print(f"❌ Evaluation failed: {batch_run.status}")

✅ Connected to: https://fndry-ws-829a01.services.ai.azure.com/api/projects/workshop-project
   Model: gpt-4.1
BASIC PROMPT RESPONSE:
The potential of AI (Artificial Intelligence) in **quality control for manufacturing** is vast and transformative. Here’s an analysis covering the main aspects:

---

## 1. **Key Benefits of AI in Manufacturing Quality Control**

### **A. Improved Defect Detection**
- **AI-powered visual inspection systems** (using machine vision + deep learning) can detect subtle defects or anomalies (scratches, misalignments, color deviations, etc.), often better than human inspectors.
- **Real-time analysis**

... (4228 chars total)

ENGINEERED PROMPT RESPONSE:
## Opportunities

**1. Enhanced Defect Detection and Accuracy**  
AI-driven vision systems can identify defects at rates exceeding 99%, versus 90-95% for humans (typical values). This boosts first-pass yield and reduces rework costs.

**2. Real-Time Quality Monitoring**  
AI algorithms provide continuous, instan

> 💡 **View results**: Click the link above to see per-row scores and aggregate metrics
> in the **new Foundry portal** (under Build → Evaluation).

---
Proceed to `04-orchestration.ipynb`.